# UR5 station — real robot

A walk-through of the `ur5_station` package on the real UR5: move the arm and
gripper, read each sensor, then record a trial with all of them together.

| Module | What it does |
|---|---|
| `ur5_station.robot_motion_client.RobotMotionClient` | arm motion: named poses, joint moves, Cartesian moves, speed |
| `ur5_station.gripper_client.GripperClient` | Robotiq 2F-85: open / close / ratio / force |
| `ur5_station.freedrive_client.FreedriveClient` | move the arm by hand |
| `ur5_station.sensors.*` | one recorder per sensor: joint states, DIGIT, GelSight Mini, camera, audio, wrist F/T |
| `ur5_station.recording_manager.RecordingManager` | starts/stops all recorders together and writes one folder per trial |

**Every motion cell moves the real arm.** Keep a hand near the emergency stop
while you run them.

## 1. Start the robot

Three steps:

### 1/3 Pendant

Power on the robot and load **`ur_drive.urp`**, but **don't press play yet**: see
[docs/robot_startup.md](../../../docs/robot_startup.md), steps 1–2.

### 2/3 Launch the bringup (PC, its own terminal)

Use a terminal that sources this workspace:

```bash
ros2 launch ur5_bringup bringup_moveit_rviz.launch.py \
  robot_ip:=182.22.22.2 reverse_ip:=182.22.22.1 gripper_com_port:=/dev/ttyUSB0
```
You should see the gripper close and open.

  This connects to the real arm (`robot_ip`), listens for the pendant's External
  Control program on this PC (`reverse_ip`), and drives the real gripper over its
  USB port. Only **one** bringup can talk to the
  robot and gripper at a time. Leave this terminal running; `Ctrl+C` in it stops
  the bringup at the end of the session.

  RViz opens and shows the arm. A back wall, a right wall and a table pad appear
  about 8 s later; MoveIt plans around them.

### 3/3 Press play (pendant)

Press **▶ Play** on the External Control program and tap **continue** if a popup
appears. The bringup terminal should print:

```text
Robot connected to reverse interface
```

Until that line appears, the arm ignores every command from the PC.

### (Optional) Check the sensors

The DIGIT and GelSight are read directly over USB by the recorders, so no launch
is needed. To find a DIGIT's serial and preview it:

```bash
ros2 run ur5_station digit_live_view --list            # prints e.g. D20503
ros2 run ur5_station digit_live_view --serial D20503   # 'q' quits, 'r' shows change from a reference
```

**Close the preview before recording.** Only one program can open the camera.

The RealSense camera and microphone are ROS nodes (optional):
```bash
ros2 launch ur5_station sensor_bringup.launch.py
```

### Attention: notebook environment

The notebook kernel inherits the environment that started Jupyter or VS Code, so
start it from a terminal in the workspace folder (wherever you cloned
`ur5_station_ws`) that sources only this workspace:

```bash
cd <path-to>/ur5_station_ws
source /opt/ros/humble/setup.bash
source install/setup.bash
jupyter lab src/ur5_station/notebooks/ur5_station_demo.ipynb   # or: code .
```

Or put the two `source` lines in `~/.bashrc`, using the full path there
(`source <path-to>/ur5_station_ws/install/setup.bash`). Otherwise the imports in
§2 fail or pick up an old copy of the packages.

## 2. Setup
_Run once per kernel session._

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── edit these before each session ──────────────────────────────────────────
DIGIT_SERIAL  = 'D20503'    # printed on the sensor; `ros2 run ur5_station digit_live_view --list`
OUTPUT_ROOT   = '~/ur5_station_data'   # where trials are saved

# Optional sensors — leave False unless the hardware is connected / launched.
ENABLE_GELSIGHT = False     # GelSight Mini over USB
ENABLE_CAMERA   = False     # RealSense, needs sensor_bringup.launch.py
ENABLE_AUDIO    = False     # ReSpeaker mic, needs sensor_bringup.launch.py
ENABLE_WRIST_FT = False     # UR wrist force/torque, real robot only
CAMERA_TOPIC    = '/camera/realsense2_camera/color/image_raw/compressed'
AUDIO_TOPIC     = '/audio'
WRIST_FT_TOPIC  = '/force_torque_sensor_broadcaster/wrench'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import csv
import math
import time
from datetime import datetime
from pathlib import Path
from threading import Thread

import cv2
import matplotlib
matplotlib.use('Agg')   # before pyplot: the inline backend breaks with matplotlib 3.5 (rcParams._get)
import matplotlib.pyplot as plt
import numpy as np
import rclpy
import yaml
from IPython.display import Image, display
from rclpy.callback_groups import ReentrantCallbackGroup
from rclpy.node import Node

from ur5_station.robot_motion_client import RobotMotionClient, ARM_JOINT_ORDER
from ur5_station.gripper_client import GripperClient
from ur5_station.freedrive_client import FreedriveClient
from ur5_station.recording_manager import RecordingManager
from ur5_station.sensors.robot_joint_state_recorder import RobotJointStateRecorder
from ur5_station.sensors.digit_recorder import DigitRecorder

OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser()
print('data ->', OUTPUT_ROOT)


def show():
    # Display the current figure in the notebook (stand-in for show() under Agg).
    import io
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=90, bbox_inches='tight')
    plt.close()
    display(Image(buf.getvalue()))

All the clients share one ROS node, spun by a background executor thread so
callbacks (joint states, action results) keep arriving while a cell blocks on a
motion. **Run this once per kernel** — running it twice raises an error; restart
the kernel instead.

In [ ]:
rclpy.init()
node = Node('ur5_station_demo')
cbg = ReentrantCallbackGroup()

executor = rclpy.executors.MultiThreadedExecutor(4)
executor.add_node(node)
spin_thread = Thread(target=executor.spin, daemon=True)
spin_thread.start()
print('node:', node.get_name())

In [ ]:
gripper = GripperClient(node, cbg)
robot = RobotMotionClient(node, cbg, velocity=0.1, acceleration=0.1, gripper=gripper)
print('robot ready:', robot.wait_until_ready(timeout=15.0))

## 3. The arm

**Coordinates.** Positions are in metres in the `world` frame; the end effector
(EEF) is `left_tool0`, the centre of the wrist flange — *not* the tip of whatever
the gripper holds. Speeds are MoveIt scaling factors in (0, 1]; `0.1` = 10 % of
the robot's maximum. Stay at or below 0.1 until a motion has been tried once.

**Known issue — the world axes are slightly rotated relative to the table.** A
move along `y` only also drifts a little in `x`. Don't rely on single-axis moves
to follow a table edge; for a straight line, record both end points (§3.4) and
move between them.

### 3.1 Read the current state

In [ ]:
joints = robot.get_current_joint_positions()
print(f'{"joint":<28} {"rad":>9} {"deg":>9}')
for name in ARM_JOINT_ORDER:
    print(f'{name:<28} {joints[name]:>9.4f} {math.degrees(joints[name]):>9.2f}')

pos, quat = robot.get_current_eef_pose()
print(f'\nEEF position  x={pos[0]:.4f}  y={pos[1]:.4f}  z={pos[2]:.4f} m')
print(f'EEF quaternion (x, y, z, w) = {[round(q, 4) for q in quat]}')

### 3.2 Named poses

Joint configurations measured at the station and stored in
`robot_motion_client.NAMED_POSES`: `home` (safe, high) and `poke_ready` (lower,
above the workspace). Start and end every session at `home`.

In [ ]:
robot.move_to_named_pose('home')

In [ ]:
robot.move_to_named_pose('poke_ready')

### 3.3 Small relative moves

`move_eef_delta(x=, y=, z=)` moves the EEF by an offset in metres, keeping its
orientation. Several axes run as **separate legs** (up first, down last), not a
diagonal. Pass `z_floor_m` on any downward move: the motion is refused if it
would end below that height.

In [ ]:
Z_FLOOR = robot.get_current_eef_pose()[0][2] - 0.05   # never more than 5 cm below here

robot.move_eef_delta(z=+0.02)                     # up 2 cm
robot.move_eef_delta(z=-0.02, z_floor_m=Z_FLOOR)  # back down

### 3.4 Go to a saved pose / straight lines

`get_current_eef_pose()` and `move_to_eef_pose()` are inverses: save a pose now,
return to it later. This is the reliable way to define a path — jog or
freedrive to each point, save it, then move between saved points.

In [ ]:
start_pose = robot.get_current_eef_pose()        # ([x, y, z], [qx, qy, qz, qw])

# Straight line to a point 2 cm higher and 1 cm over, keeping orientation:
p = start_pose[0]
robot.move_to_eef_pose([p[0] + 0.01, p[1], p[2] + 0.02])

robot.move_to_eef_pose(start_pose)               # and back

### 3.5 Joint moves

Rotate a single joint by a number of degrees — e.g. turn the wrist to face the
tactile sensor towards a camera. `move_to_joint_positions()` takes a whole
configuration (as returned by `get_current_joint_positions()`).

In [ ]:
saved_joints = robot.get_current_joint_positions()

robot.rotate_joint('left_wrist_3_joint', 15)     # +15° on the last wrist joint
robot.move_to_joint_positions(saved_joints)      # back to the saved configuration

### 3.6 Speed

In [ ]:
robot.set_speed(velocity=0.05, acceleration=0.05)   # slower for the next motions
robot.move_eef_delta(z=+0.01)
robot.move_eef_delta(z=-0.01, z_floor_m=Z_FLOOR)
robot.set_speed(velocity=0.1, acceleration=0.1)     # back to the default

### 3.7 Freedrive — move the arm by hand

Releases the brakes so the arm can be pushed into place, e.g. to teach a pose.
**Hold the arm before running** — it can sag slightly when it switches.
MoveIt motions fail while freedrive is on; the `with` block always switches back.

In [ ]:
with FreedriveClient(node, cbg):
    input('Move the arm by hand, then press Enter here ...')
    taught_pose = robot.get_current_eef_pose()
print('taught pose:', taught_pose)

## 4. The gripper

Position is a joint angle in radians (0 ≈ open, ~0.8 ≈ closed). `calibrate()`
opens fully then closes fully to learn the real range with whatever is mounted
between the fingers, after which `open_by_ratio(r)` means *fraction open*
(1 = open, 0 = closed).

**Known issue:** on this station `get_joint_pos()` returns the position the
gripper was *commanded* to, not a measurement — don't use it to detect a grasp.

In [ ]:
gripper.calibrate()
print(f'open={gripper.open_pos_rad:.3f} rad   closed={gripper.close_pos_rad:.3f} rad')

In [ ]:
gripper.open()
gripper.open_by_ratio(0.5)       # half open
print('gripper position:', round(gripper.get_joint_pos(), 3), 'rad')
gripper.open()

**Force-limited grasp.** Force only takes effect when the
gripper moves to a *new* target; on a held grasp it can be tightened, not
loosened — to grip lighter, open a little and close again at the lower force.

In [ ]:
gripper.enable_force_control()    # once per bringup
gripper.set_speed(0.03)           # m/s, slow approach
gripper.grip_with_force(30.0)     # close until contact, squeeze at ~30 N
time.sleep(1.0)
gripper.open()

## 5. Sensors, one at a time

Every recorder has the same five methods, which is what lets `RecordingManager`
(§6) drive them together:

```python
rec.health_check()        # is data arriving at the expected rate?
rec.start_recording()
rec.stop_recording()
rec.save(folder)          # writes its own sub-folder of folder
rec.validate(folder)      # re-reads what was saved and reports problems
```

This section uses them individually and saves to a scratch folder.

In [ ]:
SCRATCH = OUTPUT_ROOT / 'scratch'
SCRATCH.mkdir(parents=True, exist_ok=True)

### 5.1 Joint states

Joint angles from `/joint_states` (the arm's own encoders), saved to
`joint_states/joint_states.csv`.

In [ ]:
js = RobotJointStateRecorder(node)
print('health:', js.health_check())

js.start_recording()
robot.move_eef_delta(z=+0.02)
robot.move_eef_delta(z=-0.02, z_floor_m=Z_FLOOR)
js.stop_recording()
js_csv = js.save(SCRATCH)
print(js.validate(SCRATCH))

In [ ]:
def load_csv(path):
    with open(path) as f:
        rows = list(csv.DictReader(f))
    return {k: np.array([float(r[k]) for r in rows]) for k in rows[0]}

d = load_csv(js_csv)
t = d['timestamp'] - d['timestamp'][0]
fig, ax = plt.subplots(figsize=(9, 3.5))
for name in ARM_JOINT_ORDER:
    ax.plot(t, np.degrees(d[name] - d[name][0]), label=name.replace('left_', ''))
ax.set_xlabel('time (s)'); ax.set_ylabel('change from start (deg)')
ax.set_title('joint angles during an up/down move'); ax.legend(fontsize=8, ncol=3)
show()

### 5.2 DIGIT tactile sensor

Read straight from USB, identified by serial number. Frames are saved as JPEGs
at 320×240, ~60 fps, with a timestamp for each in `tactile_metadata.csv`.
Frames are saved as the sensor sends them (landscape); the official DIGIT library
rotates them to portrait — use `np.rot90`/`cv2.rotate` offline if needed.

In [ ]:
digit = DigitRecorder(node, serial=DIGIT_SERIAL)
print('health:', digit.health_check())

ref_path = digit.snapshot(SCRATCH, name='digit_reference', warmup_s=1.0)
plt.figure(figsize=(4, 3))
plt.imshow(cv2.cvtColor(cv2.imread(str(ref_path)), cv2.COLOR_BGR2RGB))
plt.title(f'DIGIT {DIGIT_SERIAL} — reference'); plt.axis('off'); show()

Record three seconds. **Press on the gel with a finger while this runs** so there
is something to see.

In [ ]:
digit.start_recording()
print('recording — press the gel now ...')
time.sleep(3.0)
digit.stop_recording()
digit.save(SCRATCH)
report = digit.validate(SCRATCH)
print({k: report[k] for k in ('status', 'frames_saved', 'estimated_fps', 'issues')})

In [ ]:
def load_tactile(trial_dir):
    # (timestamps, list of BGR frames) from a tactile/ folder
    folder = Path(trial_dir) / 'tactile'
    with open(folder / 'tactile_metadata.csv') as f:
        rows = list(csv.DictReader(f))
    ts = np.array([float(r['timestamp']) for r in rows])
    frames = [cv2.imread(str(folder / r['filename'])) for r in rows]
    return ts, frames


def contact_signal(frames, n_ref=10):
    # Mean absolute change from the average of the first n_ref frames — rises when
    # something presses the gel. Averaging keeps sensor noise out of the reference;
    # what remains (~2 grey levels on D20503) is the no-contact floor.
    ref = np.mean([f.astype(np.float32) for f in frames[:n_ref]], axis=0)
    return np.array([np.abs(f.astype(np.float32) - ref).mean() for f in frames])


ts, frames = load_tactile(SCRATCH)
signal = contact_signal(frames)
peak = int(signal.argmax())

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB)); axes[0].set_title('first frame')
axes[1].imshow(cv2.cvtColor(frames[peak], cv2.COLOR_BGR2RGB)); axes[1].set_title('largest change')
diff = cv2.absdiff(frames[peak], frames[0])
axes[2].imshow(np.clip(diff.astype(np.float32) * 4, 0, 255).astype(np.uint8)); axes[2].set_title('difference ×4')
for a in axes: a.axis('off')
show()

plt.figure(figsize=(9, 2.5))
plt.plot(ts - ts[0], signal)
plt.xlabel('time (s)'); plt.ylabel('mean |Δ| from reference'); plt.title('DIGIT contact signal')
show()

### 5.3 Other sensors (optional)

Same interface; each needs its hardware connected and, for the ROS-topic ones,
`sensor_bringup.launch.py` running. Enable them in the §2 config cell.

In [ ]:
optional = {}
if ENABLE_GELSIGHT:
    from ur5_station.sensors.tactile_recorder import TactileRecorder
    optional['gelsight'] = TactileRecorder(node)   # auto-finds the GelSight Mini
if ENABLE_CAMERA:
    from ur5_station.sensors.camera_recorder import CameraRecorder
    optional['camera'] = CameraRecorder(node, topic=CAMERA_TOPIC)
if ENABLE_AUDIO:
    from ur5_station.sensors.audio_recorder import AudioRecorder
    optional['audio'] = AudioRecorder(node, topic=AUDIO_TOPIC)
if ENABLE_WRIST_FT:
    from ur5_station.sensors.wrist_ft_recorder import WristFTRecorder
    optional['wrist_ft'] = WristFTRecorder(node, topic=WRIST_FT_TOPIC)

for name, rec in optional.items():
    print(name, 'health:', rec.health_check())
if not optional:
    print('no optional sensors enabled')

## 6. Recording a trial

`RecordingManager` starts and stops every recorder together and writes one folder
per trial:

```
<OUTPUT_ROOT>/<session>/trial_0000/
    joint_states/joint_states.csv     arm joint angles   (always recorded)
    tactile/frame_*.jpg               DIGIT frames
    tactile/tactile_metadata.csv      one timestamp per frame
    events.csv                        your labelled moments (mark_event)
    metadata.yaml                     whatever dict you pass in
    validation_report.yaml            checks run after the trial
```

All timestamps are Unix time in seconds on the same PC clock, so files line up by
timestamp. `mark_event('name')` drops a labelled timestamp into `events.csv` — mark
the start and end of every phase of your motion.

In [ ]:
session_dir = OUTPUT_ROOT / ('demo_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
recorder = RecordingManager(
    node, session_dir,
    tactile_recorder=digit,
    camera_recorder=optional.get('camera'),
    audio_recorder=optional.get('audio'),
    wrist_ft_recorder=optional.get('wrist_ft'),
)
print('session:', session_dir)
print('all sensors healthy:', recorder.health_check_all())

### 6.1 Run one trial

The pattern for every trial: **move to the start → start recording → mark events
around each motion → stop → validate.** Edit the motion part for your own task;
the rest stays the same.

In [ ]:
TRIAL_ID = 'trial_0001'

robot.move_to_named_pose('poke_ready')
start_pose = robot.get_current_eef_pose()
Z_FLOOR = start_pose[0][2] - 0.05

metadata = {
    'date': datetime.now().isoformat(timespec='seconds'),
    'digit_serial': DIGIT_SERIAL,
    'motion': 'up 2 cm, down 2 cm',
    'start_xyz_m': [round(v, 4) for v in start_pose[0]],
    'notes': '',
}

recorder.start_trial_recording(TRIAL_ID, metadata, overwrite='always')
recorder.mark_event('trial_start')

recorder.mark_event('up_begin')
robot.move_eef_delta(z=+0.02)
recorder.mark_event('up_end')

recorder.mark_event('down_begin')
robot.move_eef_delta(z=-0.02, z_floor_m=Z_FLOOR)
recorder.mark_event('down_end')

recorder.mark_event('trial_stop')
recorder.stop_trial_recording()

report = recorder.validate_last_trial()
print('status:', report['status'])
print(yaml.dump({k: v for k, v in report.items() if isinstance(v, dict)}, default_flow_style=False))

### 6.2 Look at what was recorded

Joint motion and the DIGIT contact signal on one time axis, with the marked events.
(In this demo nothing touches the gel, so the tactile signal stays flat — in a
real trial it rises when the held tool makes contact.)

In [ ]:
trial_dir = recorder.trial_dir
print('files:')
for p in sorted(trial_dir.rglob('*')):
    if p.is_file() and not p.name.startswith('frame_'):
        print('  ', p.relative_to(trial_dir))
print('   tactile/frame_*.jpg ×', len(list((trial_dir / 'tactile').glob('frame_*.jpg'))))

with open(trial_dir / 'events.csv') as f:
    events = [(float(r['timestamp']), r['event']) for r in csv.DictReader(f)]
t0 = events[0][0]

d = load_csv(trial_dir / 'joint_states' / 'joint_states.csv')
ts, frames = load_tactile(trial_dir)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for name in ARM_JOINT_ORDER:
    ax1.plot(d['timestamp'] - t0, np.degrees(d[name] - d[name][0]), label=name.replace('left_', ''))
ax1.set_ylabel('joint change (deg)'); ax1.legend(fontsize=7, ncol=3)
ax2.plot(ts - t0, contact_signal(frames))
ax2.set_ylabel('DIGIT mean |Δ|'); ax2.set_xlabel('time since trial_start (s)')
for t, name in events:
    for ax in (ax1, ax2):
        ax.axvline(t - t0, color='grey', lw=0.8, ls='--')
    ax1.text(t - t0, ax1.get_ylim()[1], name, rotation=90, va='top', fontsize=7)
show()

In [ ]:
# DIGIT frame closest to each event
fig, axes = plt.subplots(1, len(events), figsize=(2.6 * len(events), 2.4))
for ax, (t, name) in zip(axes, events):
    i = int(np.abs(ts - t).argmin())
    ax.imshow(cv2.cvtColor(frames[i], cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=8); ax.axis('off')
show()

## 7. Writing your own motion

Put the motion in a function that takes the clients and the recorder, marks an
event around each phase, and returns what it actually did. Keep it in your own
module (e.g. `my_task/motions.py`) rather than editing `ur5_station`, so the
station code stays shared. A template:

```python
def press_and_release(robot, recorder, depth_m, z_floor_m):
    # Lower the EEF by depth_m, hold, and lift back. Returns the lowest EEF z reached.
    recorder.mark_event('press_begin')
    robot.move_eef_delta(z=-depth_m, z_floor_m=z_floor_m)
    lowest_z = robot.get_current_eef_pose()[0][2]
    recorder.mark_event('press_end')

    time.sleep(1.0)                      # hold

    recorder.mark_event('release_begin')
    robot.move_eef_delta(z=+depth_m)
    recorder.mark_event('release_end')
    return lowest_z
```

Rules that keep the robot and the data safe:
- Every downward motion gets a `z_floor_m`.
- Run a new motion first at `velocity=0.05` with a hand on the emergency
  stop, and only speed it up once it has run cleanly.
- Check the return value of motion calls (`True` = reached the target) and stop
  the trial if one is `False`, rather than continuing from the wrong place.

## 8. Shutdown
_Run at the end of the session; restart the kernel before running §2 again._

In [ ]:
robot.move_to_named_pose('home')
gripper.open()

In [ ]:
executor.shutdown()
rclpy.shutdown()
print('shutdown complete')